# Phase 2: Exploratory Data Analysis (EDA)
**E-commerce Sales dataset** (Kaggle: berkayalan/ecommerce-sales-dataset)

Files: `customer_details.csv` and `basket_details.csv`.

The raw data is only **read** here, never modified. Cleaning happens in the next notebook, based on the problems found below.

## 1. Setup and Extract

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

# Looks in data/raw first, then in the notebook's own folder
DATA_DIR = Path("data/raw") if Path("data/raw/customer_details.csv").exists() else Path(".")

customer_raw = pd.read_csv(DATA_DIR / "customer_details.csv")
basket_raw = pd.read_csv(DATA_DIR / "basket_details.csv")

# Work on copies so the raw frames stay untouched
customer = customer_raw.copy()
basket = basket_raw.copy()
print("Loaded from:", DATA_DIR.resolve())

## 2. Basic dataset exploration

### 2.1 Customer data

In [ ]:
print("Rows, columns:", customer.shape)
print("Columns:", list(customer.columns))
customer.head()

In [ ]:
customer.dtypes

In [ ]:
customer.describe(include="all").T

In [ ]:
customer.nunique()

In [ ]:
customer.sample(5, random_state=42)

### 2.2 Basket data

In [ ]:
print("Rows, columns:", basket.shape)
print("Columns:", list(basket.columns))
basket.head()

In [ ]:
basket.dtypes

In [ ]:
basket.describe(include="all").T

In [ ]:
basket.nunique()

In [ ]:
basket.sample(5, random_state=42)

### 2.3 Key columns

In [ ]:
print("Customers in customer file :", customer["customer_id"].nunique())
print("Customers in basket file   :", basket["customer_id"].nunique())
print("Unique products            :", basket["product_id"].nunique())
print("Basket rows                :", len(basket))
print("Total units (basket_count) :", basket["basket_count"].sum())

## 3. Data quality analysis

### 3.1 Missing values

In [ ]:
print("Customer:")
print(customer.isna().sum())
print("\nBasket:")
print(basket.isna().sum())

### 3.2 Duplicates

In [ ]:
print("Customer fully duplicated rows        :", customer.duplicated().sum())
print("Customer duplicated customer_id       :", customer["customer_id"].duplicated().sum())
print("Basket fully duplicated rows          :", basket.duplicated().sum())
print("Basket duplicated (cust, product, date):",
      basket.duplicated(["customer_id", "product_id", "basket_date"]).sum())

### 3.3 Incorrect data types

In [ ]:
parsed_dates = pd.to_datetime(basket["basket_date"], errors="coerce")
print("basket_date dtype   :", basket["basket_date"].dtype, "-> should be datetime")
print("customer_age dtype  :", customer["customer_age"].dtype, "-> should be integer")
print("Unparseable dates   :", parsed_dates.isna().sum())

### 3.4 Inconsistent categories (`sex`)

In [ ]:
customer["sex"].value_counts(dropna=False)

`kvkktalepsilindi` looks like a Turkish data-deletion marker and `UNKNOWN` is a placeholder. Both need standardising.

### 3.5 Invalid values and outliers: `customer_age`

In [ ]:
age = customer["customer_age"]
print(age.describe())
print()
print("Age < 0                         :", (age < 0).sum())
print("Age 0-17                        :", ((age >= 0) & (age < 18)).sum())
print("Age 18-90 (plausible)           :", ((age >= 18) & (age <= 90)).sum())
print("Age > 90                        :", (age > 90).sum())
print("Age > 1900 (maybe a birth year) :", (age > 1900).sum())

In [ ]:
# Most common implausible ages
age[(age < 18) | (age > 90)].value_counts().head(10)

### 3.6 Invalid values and outliers: `tenure` and `basket_count`

In [ ]:
print(customer["tenure"].describe())
print("Tenure <= 0:", (customer["tenure"] <= 0).sum())

In [ ]:
def iqr_outliers(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return s[(s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)]

print("Tenure outliers (IQR)       :", len(iqr_outliers(customer["tenure"])))
print("basket_count outliers (IQR) :", len(iqr_outliers(basket["basket_count"])))
basket["basket_count"].value_counts().sort_index()

### 3.7 Referential integrity between the two files

In [ ]:
in_cust = basket["customer_id"].isin(customer["customer_id"])
no_orders = ~customer["customer_id"].isin(basket["customer_id"])

print(f"Basket rows with a matching customer : {in_cust.sum()} ({in_cust.mean():.2%})")
print(f"Basket rows with NO matching customer: {(~in_cust).sum()} ({(~in_cust).mean():.2%})")
print(f"Distinct orphan customer_ids         : {basket.loc[~in_cust, 'customer_id'].nunique()}")
print(f"Customers with no basket rows        : {no_orders.sum()} ({no_orders.mean():.2%})")

**Important:** almost none of the basket customers exist in the customer file. This decides how the database is designed later (see the plan).

## 4. Exploratory analysis

In [ ]:
# Working copy with parsed dates (raw frame is unchanged)
basket["basket_date"] = parsed_dates
basket["day_of_week"] = basket["basket_date"].dt.day_name()

print("From:", basket["basket_date"].min().date(), " To:", basket["basket_date"].max().date())
print("Days covered:", basket["basket_date"].nunique())

### 4.1 Units and rows per day

In [ ]:
daily = (
    basket.groupby("basket_date")
    .agg(rows=("product_id", "size"),
         units=("basket_count", "sum"),
         customers=("customer_id", "nunique"))
    .sort_index()
)
print("Busiest day :", daily["units"].idxmax().date(), "->", daily["units"].max(), "units")
print("Quietest day:", daily["units"].idxmin().date(), "->", daily["units"].min(), "units")
daily

### 4.2 Day-of-week pattern

In [ ]:
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
dow = (basket.groupby("day_of_week")["basket_count"]
       .agg(rows="size", units="sum", avg_units_per_row="mean")
       .reindex(order).round(2))
dow

### 4.3 Product performance

In [ ]:
prod = (basket.groupby("product_id")["basket_count"]
        .agg(units="sum", rows="size")
        .sort_values("units", ascending=False))

print("Top 10 products by units:")
display(prod.head(10))
print("Products appearing in exactly one row:", (prod["rows"] == 1).sum(), "of", len(prod))
print(f"Top 10 products' share of all units: {prod['units'].head(10).sum() / prod['units'].sum():.2%}")

### 4.4 Customers by sex and age group (plausible ages only)

In [ ]:
valid = customer.loc[customer["customer_age"].between(18, 90)].copy()
valid["age_group"] = pd.cut(valid["customer_age"], [17, 24, 34, 44, 54, 64, 90],
                            labels=["18-24", "25-34", "35-44", "45-54", "55-64", "65+"])

print(customer["sex"].value_counts(), "\n")
print(valid["age_group"].value_counts().sort_index(), "\n")
print("Average tenure by age group:")
print(valid.groupby("age_group", observed=True)["tenure"].mean().round(1), "\n")
print("Average tenure by sex:")
print(customer.groupby("sex")["tenure"].mean().round(1))

### 4.5 Heaviest buyers

In [ ]:
buyers = (basket.groupby("customer_id")["basket_count"]
          .agg(units="sum", rows="size")
          .sort_values("units", ascending=False))
print("Average units per customer      :", round(buyers["units"].mean(), 2))
print("Customers with more than 1 row  :", (buyers["rows"] > 1).sum())
buyers.head(10)

### 4.6 `iloc` and `loc` examples

In [ ]:
print("First 3 rows, first 4 columns (iloc):")
display(basket.iloc[:3, :4])
print("Rows with 5+ units (loc):")
display(basket.loc[basket["basket_count"] >= 5, ["customer_id", "product_id", "basket_count"]].head())

## 5. Visualisations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

customer.loc[customer["customer_age"].between(0, 120), "customer_age"].plot.hist(bins=40, ax=axes[0])
axes[0].set_title("Customer age (0-120 only)")
axes[0].set_xlabel("Age")

customer["sex"].value_counts().plot.bar(ax=axes[1])
axes[1].set_title("Customers by sex (raw categories)")
axes[1].tick_params(axis="x", rotation=20)

customer["tenure"].plot.hist(bins=30, ax=axes[2])
axes[2].set_title("Tenure distribution")
axes[2].set_xlabel("Tenure")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))

daily["units"].plot(marker="o", ax=axes[0])
axes[0].set_title("Units sold per day")
axes[0].set_xlabel("")

dow["units"].plot.bar(ax=axes[1])
axes[1].set_title("Units by day of week")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))

prod["units"].head(10).sort_values().plot.barh(ax=axes[0])
axes[0].set_title("Top 10 products by units")
axes[0].set_xlabel("Units")

basket["basket_count"].value_counts().sort_index().plot.bar(ax=axes[1])
axes[1].set_title("basket_count distribution")
axes[1].set_ylabel("Rows")

plt.tight_layout()
plt.show()

## 6. EDA summary: problems to fix in the cleaning phase

In [ ]:
age = customer["customer_age"]
print(f"1. customer_age invalid: {(age < 18).sum()} below 18 (incl. {(age < 0).sum()} negative), {(age > 90).sum()} above 90.")
print(f"2. sex has {customer['sex'].nunique()} categories; {(~customer['sex'].isin(['Male', 'Female'])).sum()} rows are not Male/Female.")
print("3. basket_date is text; customer_age is float.")
print(f"4. {(~in_cust).sum()} basket rows have no matching customer ({(~in_cust).mean():.2%}); {no_orders.sum()} customers have no baskets.")
print("5. No price or category columns: sales can only be measured in units.")
print(f"6. Data covers only {daily.shape[0]} days: use daily/weekday trends, not monthly.")

### Findings to write up in the report
Fill these in from the outputs above:
- Busiest and quietest day:
- Share of units from the top 10 products:
- Typical `basket_count` (most rows have 2 units):
- Main data quality problems (see summary above):